# One pass per question: Signpost with every option in one input

**Part 4b of "LLMs the Hard Way"** (branch `signpost`), next to `03_lora_signpost.ipynb`. Signpost reads the conversation once **per option**: 15 agents means 15 encoder passes, and its latency grows with every agent. Here every option goes into **one** input, the encoder runs **once**, and one shared head scores each option at its own tokens:

```text
<history> [SEP] current agent: X [SEP] question: Q [SEP] option: Billing Agent. Handles… [SEP] option: Returns Agent. … [SEP] …
   → ModernBERT-large + LoRA (one pass) → the mean of each option's tokens → the same head on each → softmax over the N options
```

The options also see each other inside the encoder, so the model can compare them directly.

**What differs from Signpost** (the recipe of run 28, the best of runs 26–29; Cell 9 has the others):
- `MAX_LEN` 1,536 tokens, enough for a question with 50 options or a chat with 15 agents. If the input is too long, the oldest history is dropped; options never are.
- **Training data:** all of Signpost's routing and question data, plus **2,500 hand-written many-option questions** (957 individually written texts, 2–50 options).
- **Distractor agents:** each time a routing question is used, agents from other businesses are added (never generic names or names sharing a word with its own agents): 10–15 agents in total for 2,500 of the routing questions, 5–9 for the rest.
- **Plain cross-entropy for score questions** (run 28; the ordinal loss of Signpost hurt here).
- **Training batches by token budget**, so long inputs and short ones aren't padded together.
- LoRA through `peft`, as in `03`.

**Tests:** every set of `03` (E10, E11, banking, E12, both independent sets), plus three **fresh many-option tests** written for this model (162 questions with 10–50 options using words that never appear in training; 24 conversations and 72 first messages at 12 new businesses with 9–15 agents), and an **option-order check** (the same first messages with the agents reversed).

**Compared with** the published Signpost and the 2B Strands decider: their numbers are measured results on the same items (Signpost in `03` and from the Hub, the decider in `04`), so neither runs here. At the end the model is saved and checked with its own standalone loader, `signpost/signpost_one_pass.py`.

## Cell 1 — Setup

| Setting | Default | Meaning |
|---|---|---|
| `MAX_LEN` | 1536 | tokens per input: history + question + every option |
| `MAX_OPTIONS` | 50 | more options than this raises an error |
| `AUG_MAX_AGENTS` | 15 | routing questions are trained with up to this many agents |
| `MANY_AGENT_ROUTES` | 2500 | routing questions trained with 10–15 agents each time (the others with 5–9) |
| `TOKEN_BUDGET`, `MAX_BATCH` | 8000, 16 | a training batch holds about this many tokens, padding included, and at most this many questions |
| `EPOCHS`, `LORA_R`, `LORA_ALPHA`, `LR`, `HEAD_LR` | 2, 16, 32, 2e-4, 1e-3 | as in Signpost |
| `SEED` | 42 | the first seed; the collapse guard moves on by +1000 per retry |
| `SAVE_MODEL` | 1 | merge LoRA, save `signpost-one-pass/` + zip in the notebook's working folder, reload with the loader and check |
| `LORA_SMOKE` | 0 | 1 = a tiny code check on ModernBERT-base |

Runs on Kaggle with a T4 and Internet on, in about 40 minutes of training plus the tests.

In [1]:
# Cell 1 — Setup: packages, settings, device, progress log

!pip uninstall -y -q torchao                 # Kaggle ships torchao 0.10; peft refuses versions below 0.16
!pip install -q "transformers==5.17.0" "peft==0.21.2"

import copy, gc, json, os, random, shutil, statistics, subprocess, sys, time
from collections import Counter
from pathlib import Path

import peft
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers

SMOKE = os.environ.get("LORA_SMOKE") == "1"
ENCODER = "answerdotai/ModernBERT-base" if SMOKE else "answerdotai/ModernBERT-large"
LORA_R, LORA_ALPHA, LORA_DROPOUT = int(os.environ.get("LORA_R", "16")), float(os.environ.get("LORA_ALPHA", "32")), 0.1
LR, HEAD_LR = float(os.environ.get("LR", "2e-4")), float(os.environ.get("HEAD_LR", "1e-3"))
EPOCHS = int(os.environ.get("EPOCHS", "1" if SMOKE else "2"))
MAX_LEN, MAX_OPTIONS = int(os.environ.get("MAX_LEN", "1536")), int(os.environ.get("MAX_OPTIONS", "50"))
AUG_MAX_AGENTS, MANY_AGENT_ROUTES = int(os.environ.get("AUG_MAX_AGENTS", "15")), int(os.environ.get("MANY_AGENT_ROUTES", "2500"))
TOKEN_BUDGET, MAX_BATCH = int(os.environ.get("TOKEN_BUDGET", "8000")), int(os.environ.get("MAX_BATCH", "16"))
SEED = int(os.environ.get("SEED", "42"))
SAVE_MODEL = os.environ.get("SAVE_MODEL", "1") == "1"
SEP, ROUTE_Q = " [SEP] ", "Which agent should handle the user's latest message?"

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"
RUN_LOG = Path("one_pass_run.log")


def log(message):
    line = f"[{time.strftime('%H:%M:%S')}] {message}"
    print(line, flush=True)
    with open(RUN_LOG, "a") as fh:
        fh.write(line + "\n")


def gpu():
    return f"GPU {torch.cuda.memory_allocated(DEVICE) / 1e9:.1f} GB (peak {torch.cuda.max_memory_allocated(DEVICE) / 1e9:.1f})" if AMP else "CPU"


log(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | {DEVICE}"
    + (f" ({torch.cuda.get_device_name(DEVICE)})" if AMP else "") + f" | {ENCODER}" + (" | SMOKE CHECK" if SMOKE else ""))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 93.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 835.4/835.4 kB 38.8 MB/s eta 0:00:00
[07:34:29] torch 2.11.0+cu128 | transformers 5.17.0 | peft 0.21.2 | cuda:0 (Tesla T4) | answerdotai/ModernBERT-large


---
# Data

## Cell 2 — Training data with distractor agents, and the tests

The items are Signpost's format (`{task, messages, current, question, options, answer}`) plus the 2,500 many-option questions; the dev sets are split the same way as in `03` (10% of routing domains, 10% of question texts), so dev numbers stay comparable. Routing questions keep their own 5 agents here: distractors are added during training, a new random set each time. `dev_route_many` is the routing dev set with 15 agents, fixed, for the training log.

**Expect.** About 6,300 routing and 11,800 question items for training, 2,500 of the routing ones marked for 10–15 agents, and the size of every test.

In [2]:
# Cell 2a — Training items, dev splits, distractor agents

DATA_BRANCH = "signpost"
DATA_DIR = Path("data/modernbert_router")
if not DATA_DIR.exists():                                   # on Kaggle / Colab: clone this branch (an existing clone is refreshed)
    repo = Path("llms-the-hard-way")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", DATA_BRANCH,
                        "https://github.com/skamalj/llms-the-hard-way.git", str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", DATA_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo), "checkout", "-B", DATA_BRANCH, "FETCH_HEAD"], check=True)
    DATA_DIR = repo / "data" / "modernbert_router"
load = lambda p: json.load(open(DATA_DIR / p))

domains = {**load("generic/domains.json"), **load("realistic_train/domains.json")}
route_items = [{"task": "routing", "domain": ex["domain"], "messages": ex["messages"], "current": ex["current_agent"],
                "question": ROUTE_Q, "options": domains[ex["domain"]]["agents"], "answer": ex["target_agent"]}
               for ex in load("generic/conversations.json") + load("realistic_train/conversations.json")]
choice_items = [{"task": it["type"], "domain": it["domain"], "messages": [{"role": "user", "content": it["state"]}], "current": None,
                 "question": it["question"], "options": {o: "" for o in it["options"]}, "answer": it["answer"], "state": it["state"]}
                for it in load("choice_train/items.json")["items"] + load("many_train/items.json")["items"]]     # + 2,500 many-option

rng = random.Random(SEED)
route_domains = sorted({x["domain"] for x in route_items}); rng.shuffle(route_domains)
dev_domains = set(route_domains[: max(1, len(route_domains) // 10)])
texts = sorted({x["state"] for x in choice_items}); rng.shuffle(texts)
dev_texts = set(texts[: len(texts) // 10])
train_route, dev_route = [x for x in route_items if x["domain"] not in dev_domains], [x for x in route_items if x["domain"] in dev_domains]
train_choice, dev_choice = [x for x in choice_items if x["state"] not in dev_texts], [x for x in choice_items if x["state"] in dev_texts]

GENERIC = {"general", "support", "help", "customer", "customers", "billing", "payment", "payments", "account", "accounts",
           "booking", "bookings", "enquiries", "service", "services", "care", "desk", "team"}


def name_words(name):
    return {w.strip(",.&'").lower() for w in name.split()} - {"agent", "and", "&", ""}


DISTRACTORS = [(n, d, dom) for dom, info in domains.items() for n, d in info["agents"].items() if not name_words(n) & GENERIC]


def add_distractors(item, rng, n_total):
    """The item with its own agents plus agents from other businesses, up to n_total; none shares a name word with its own."""
    own = set().union(*(name_words(n) for n in item["options"]))
    pool = [(n, d) for n, d, dom in DISTRACTORS if dom != item["domain"] and not name_words(n) & own]
    return {**item, "options": {**item["options"], **dict(rng.sample(pool, max(0, min(len(pool), n_total - len(item["options"])))))}}


for x in random.Random(SEED).sample(train_route, min(MANY_AGENT_ROUTES, len(train_route))):
    x["many_agents"] = True                                 # trained with 10-15 agents each time; the others with 5-9
dev_route_many = [add_distractors(x, random.Random(SEED + k), AUG_MAX_AGENTS) for k, x in enumerate(dev_route)]
if SMOKE:
    train_route, dev_route, train_choice, dev_choice = train_route[::400], dev_route[::100], train_choice[::400], dev_choice[::100]
    dev_route_many = dev_route_many[::100]

for name, xs in [("train routing", train_route), ("train questions", train_choice), ("dev routing", dev_route), ("dev questions", dev_choice)]:
    print(f"{name:15s}: {len(xs):5d} | {dict(Counter(x['task'] for x in xs))}")
print(f"routing items trained with 10-15 agents: {sum(x.get('many_agents', False) for x in train_route)} | "
      f"many-option questions: {sum(x['domain'].startswith('many/') for x in train_choice)} | distractor agents: {len(DISTRACTORS)}")

Cloning into 'llms-the-hard-way'...


train routing  :  6319 | {'routing': 6319}
train questions: 11730 | {'noul': 5000, 'choice': 4788, 'score': 1942}
dev routing    :   531 | {'routing': 531}
dev questions  :  1370 | {'noul': 620, 'choice': 544, 'score': 206}
routing items trained with 10-15 agents: 2500 | many-option questions: 2279 | distractor agents: 309


In [3]:
# Cell 2b — The tests: 03's sets, plus the many-option and many-agent ones

real_convs = load("realistic/conversations.json")                                      # E10
ft = load("first_turn/messages.json")
e11 = [{**m, "agents": ft["companies"][m["company"]]} for m in ft["messages"]]          # E11
banking = load("banking/benchmark.json") + load("banking/extended.json")
e12 = load("choice_test/items.json")["items"]
ind_q, ind_r, ind_f = load("independent/questions.json")["items"], load("independent/routing.json"), load("independent/first_turn.json")["messages"]
ind2_q, ind2_r, ind2_f = (load("independent2/questions.json")["items"], load("independent2/routing.json"),
                          load("independent2/first_turn.json")["messages"])
many_q, many_r, many_f = load("many_test/questions.json")["items"], load("many_test/routing.json"), load("many_test/first_turn.json")["messages"]
BANKING_AGENTS = {                                          # exactly as in 02_frozen_router.ipynb, Cell 9, and 03
    "Payments Agent": "Handles bank transfers, failed or pending payments, transaction status, payment charges and "
                      "duplicate charges, refunds for transactions, and settlement problems where money was sent but "
                      "not received. Does not handle card servicing such as blocking, replacing or activating cards.",
    "Card Support Agent": "Handles debit and credit card servicing: card activation, blocking and unblocking, lost or "
                          "stolen cards, card replacement and delivery, card limits, card PIN, card security and "
                          "compromised card details, and card-specific transactions.",
    "Investment Agent": "Handles mutual funds, SIPs, investments, portfolio allocation, investment transactions, "
                        "redemptions and investment planning.",
    "Insurance Agent": "Handles insurance policies, premiums, renewals, claims, policy expiry and policy servicing.",
    "General Customer Support Agent": "Handles general account questions, profile and contact details, issues that do "
                                      "not clearly belong to a specialist, and routing customers to the right agent.",
}
companies = sorted({c["company"] for c in real_convs})
TEST_COMPANIES = set(companies) - set(companies[::2])         # the 6 test companies of 02, Cell 16, and 03
if SMOKE:
    real_convs, e11, banking, e12 = real_convs[:2], e11[::20], banking[:6], e12[::20]
    ind_q, ind_r, ind_f, ind2_q, ind2_r, ind2_f = ind_q[::30], ind_r[:2], ind_f[::20], ind2_q[::20], ind2_r[:2], ind2_f[::10]
    many_q, many_r, many_f = many_q[::20], many_r[:2], many_f[::12]

turns = lambda convs: sum(len(c["user_turns"]) for c in convs)
print(f"E10 {turns(real_convs)} turns | E11 {len(e11)} | banking {len(banking)} | E12 {len(e12)} | independent {len(ind_q)} questions, "
      f"{turns(ind_r)} turns, {len(ind_f)} first | independent 2 {len(ind2_q)} questions, {turns(ind2_r)} turns, {len(ind2_f)} first")
print(f"many-option: {len(many_q)} questions ({min(len(q['options']) for q in many_q)}-{max(len(q['options']) for q in many_q)} options) | "
      f"many-agent: {turns(many_r)} turns, {len(many_f)} first messages ({min(m['n_agents'] for m in many_f)}-{max(m['n_agents'] for m in many_f)} agents)")

E10 193 turns | E11 100 | banking 225 | E12 170 | independent 300 questions, 172 turns, 200 first | independent 2 200 questions, 70 turns, 50 first
many-option: 162 questions (10-50 options) | many-agent: 120 turns, 72 first messages (9-15 agents)


---
# The model

## Cell 3 — One input, one pass, one head per option

The encoder, LoRA (`peft`, r 16 on `Wqkv`, `Wo`, `Wi`) and head (`Linear(H, 256) → GELU → Linear(256, 1)`) are Signpost's. What changes is `scores`:
1. `fit` builds one text per question with the options last, each marked `option:`, records each option's **character span**, and drops the oldest messages until the text fits `MAX_LEN`.
2. One encoder pass over the batch. The tokenizer's offset mapping, compared with every span at once, gives a mask `[questions, options, tokens]`: which tokens belong to which option.
3. Each option's vector is the **mean of its tokens** (`mask @ hidden / count`); the same head scores every option. Missing options (questions with fewer options than others in the batch) get −10,000, so the softmax ignores them.

In [4]:
# Cell 3 — The one-pass model: LoRA (peft), the input with option spans, scores for every option at once

from peft import LoraConfig, get_peft_model
from transformers import AutoModel, AutoTokenizer


class OnePassModel:
    def __init__(self):
        self.tok = AutoTokenizer.from_pretrained(ENCODER)
        self.tok.padding_side = "right"
        enc = AutoModel.from_pretrained(ENCODER, dtype=torch.float32).to(DEVICE)
        self.H = enc.config.hidden_size
        self.enc = get_peft_model(enc, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                                  target_modules=["Wqkv", "Wo", "Wi"]))
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(self.H, 256), nn.GELU(), nn.Dropout(0.1), nn.Linear(256, 1)).to(DEVICE)

    def lora_params(self):
        return [p for p in self.enc.parameters() if p.requires_grad]

    def train(self, mode=True):
        self.enc.train(mode); self.head.train(mode)

    @staticmethod
    def build(item, drop=0):
        """messages SEP current agent SEP question SEP option: A SEP option: B ... -> (text, character span of each option)."""
        text = SEP.join([f"{m['role']}: {m['content']}" for m in item["messages"][drop:]]
                        + [f"current agent: {item['current'] or 'none'}", f"question: {item['question']}"])
        spans = []
        for name, desc in item["options"].items():
            seg = f"{name}. {desc}" if desc else name
            text += SEP + "option: "
            spans.append((len(text), len(text) + len(seg)))
            text += seg
        return text, spans

    def fit(self, item):
        """Drop the oldest messages until the input fits MAX_LEN; the latest message and every option always stay."""
        assert len(item["options"]) <= MAX_OPTIONS, f"{len(item['options'])} options > MAX_OPTIONS={MAX_OPTIONS}"
        for drop in range(len(item["messages"])):
            text, spans = self.build(item, drop)
            if len(self.tok(text)["input_ids"]) <= MAX_LEN:
                return text, spans
        raise ValueError(f"{len(item['options'])} options don't fit in MAX_LEN={MAX_LEN}, even with only the latest message")

    def scores(self, items):
        """-> [items, most options] scores (missing options -1e4): one encoder pass for the whole batch."""
        built = [self.fit(it) for it in items]
        enc = self.tok([t for t, _ in built], padding=True, return_offsets_mapping=True, return_tensors="pt")
        offsets = enc.pop("offset_mapping")                                            # [B, T, 2] characters of each token
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=AMP):
            hidden = self.enc(input_ids=enc["input_ids"].to(DEVICE), attention_mask=enc["attention_mask"].to(DEVICE)).last_hidden_state
        K = max(len(sp) for _, sp in built)
        spans = torch.tensor([sp + [(0, 0)] * (K - len(sp)) for _, sp in built])     # [B, K, 2]; missing options are empty
        s, e = offsets[..., 0], offsets[..., 1]                                        # [B, T]
        real = enc["attention_mask"].bool() & (e > s)                                  # real, non-special tokens
        member = (real[:, None] & (s[:, None] < spans[..., 1:]) & (e[:, None] > spans[..., :1])).float().to(DEVICE)   # [B, K, T]
        counts = member.sum(-1)                                                        # [B, K] tokens per option
        assert int((counts > 0).sum()) == sum(len(sp) for _, sp in built), "an option got no tokens"
        pooled = (member @ hidden.float()) / counts.clamp(min=1).unsqueeze(-1)         # [B, K, H] mean of each option's tokens
        return self.head(pooled).squeeze(-1).masked_fill(counts == 0, -1e4)

    def probs(self, item):
        """Probabilities over one item's options, in order; no gradient."""
        with torch.no_grad():
            return torch.softmax(self.scores([item]), dim=1)[0].float().cpu()

---
# Training

## Cell 4 — Batches by token budget, distractors, the training loop

- **Batches:** questions sorted by estimated length (routing ones counted at their largest, 15 agents), filled up to `TOKEN_BUDGET` tokens or `MAX_BATCH` questions.
- **Each time a routing question is used** it gets a new random set of distractor agents (10–15 in total for the marked ones, 5–9 for the rest); then every question's options are shuffled, so position can't be learned.
- **Loss:** plain cross-entropy over each question's options, score questions included. AdamW (adapters and head at their own learning rates), 6% warm-up then linear decay, gradient clipping at 1.0, fp16 mixed precision.
- **Best epoch** on the mean of dev routing and dev questions; the dev routing with 15 agents is logged too. **Collapse guard** as in `03`.

In [5]:
# Cell 4a — Token-budget batches, dev accuracy and the collapse guard

def est_tokens(it):
    """Rough token count before tokenizing (4 characters a token); routing questions at their largest (AUG_MAX_AGENTS agents)."""
    chars = sum(len(m["content"]) + 12 for m in it["messages"]) + len(it["question"]) + 40
    chars += sum(len(n) + len(d) + 12 for n, d in it["options"].items())
    if it["task"] == "routing":
        chars += max(0, AUG_MAX_AGENTS - len(it["options"])) * 130
    return min(MAX_LEN, chars // 4 + 8)


def make_batches(items):
    """Questions of similar length together; at most TOKEN_BUDGET tokens (padding included) and MAX_BATCH questions a batch."""
    batches, cur, longest = [], [], 0
    for it in sorted(items, key=est_tokens):
        n = est_tokens(it)
        if cur and (max(longest, n) * (len(cur) + 1) > TOKEN_BUDGET or len(cur) >= MAX_BATCH):
            batches.append(cur)
            cur, longest = [], 0
        cur.append(it)
        longest = max(longest, n)
    return batches + ([cur] if cur else [])


def dev_accuracy(model, items, batch=8):
    model.train(False)
    ok = 0
    with torch.no_grad():
        for s0 in range(0, len(items), batch):
            chunk = items[s0:s0 + batch]
            ok += sum(list(it["options"])[k] == it["answer"] for it, k in zip(chunk, model.scores(chunk).argmax(1).tolist()))
    return ok / max(1, len(items)), ok


class Collapsed(Exception):
    """Training collapsed (run 22: the question loss stayed at chance after the warm-up peak); retry with another seed."""


COLLAPSE_DEV = {"routing": 0.95, "questions": 0.85}

In [6]:
# Cell 4b — The training loop

def train_model(model, seed):
    batches = make_batches(train_route) + make_batches(train_choice)
    params = model.lora_params() + list(model.head.parameters())
    opt = torch.optim.AdamW([{"params": model.lora_params(), "lr": LR}, {"params": model.head.parameters(), "lr": HEAD_LR}], weight_decay=0.01)
    total = EPOCHS * len(batches)
    warm = max(1, int(0.06 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min((s + 1) / warm, max(0.0, (total - s) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    best, step, t0 = (-1.0, None, 0), 0, time.perf_counter()
    log(f"training: {len(batches)} batches per epoch ({len(train_route)} routing + {len(train_choice)} questions), {total} steps")
    for epoch in range(1, EPOCHS + 1):
        model.train(True)
        order = list(range(len(batches)))
        random.Random(seed + epoch).shuffle(order)
        aug = random.Random(seed * 7 + epoch)
        run = {"routing": [], "questions": []}
        for bi in order:
            items = [add_distractors(it, aug, aug.randint(10, AUG_MAX_AGENTS) if it.get("many_agents") else aug.randint(len(it["options"]), 9))
                     if it["task"] == "routing" else dict(it) for it in batches[bi]]      # a new set of extra agents every time
            for it in items:                                    # options shuffled, so position can't be learned
                names = list(it["options"]); random.shuffle(names); it["options"] = {n: it["options"][n] for n in names}
            target = torch.tensor([list(it["options"]).index(it["answer"]) for it in items], device=DEVICE)
            loss = F.cross_entropy(model.scores(items), target)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update(); sched.step()
            run["routing" if items[0]["task"] == "routing" else "questions"].append(loss.item())
            step += 1
            if step % 100 == 0 or step == total:
                el = time.perf_counter() - t0
                log(f"  epoch {epoch} step {step}/{total} | loss routing {statistics.mean(run['routing'][-50:] or [0]):.4f} "
                    f"questions {statistics.mean(run['questions'][-50:] or [0]):.4f} | {step / el:.2f} steps/s | "
                    f"~{(total - step) / (step / el) / 60:.1f} min left | {gpu()}")
        (acc_r, ok_r), (acc_m, ok_m), (acc_c, ok_c) = dev_accuracy(model, dev_route), dev_accuracy(model, dev_route_many), dev_accuracy(model, dev_choice)
        msg = (f"dev routing {ok_r}/{len(dev_route)} ({acc_r:.1%}) | with {AUG_MAX_AGENTS} agents {ok_m}/{len(dev_route_many)} ({acc_m:.1%}) | "
               f"dev questions {ok_c}/{len(dev_choice)} ({acc_c:.1%})")
        log(f"epoch {epoch} done | {msg}")
        if epoch == 1 and (acc_r < COLLAPSE_DEV["routing"] or acc_c < COLLAPSE_DEV["questions"]):
            raise Collapsed(f"{msg} after epoch 1 (thresholds {COLLAPSE_DEV})")
        if (acc_r + acc_c) / 2 > best[0]:
            best = ((acc_r + acc_c) / 2, {"lora": {n: p.detach().clone() for n, p in model.enc.named_parameters() if p.requires_grad},
                                          "head": copy.deepcopy(model.head.state_dict())}, epoch)
    params_by_name = dict(model.enc.named_parameters())
    for n, v in best[1]["lora"].items():
        params_by_name[n].data.copy_(v)
    model.head.load_state_dict(best[1]["head"])
    minutes = (time.perf_counter() - t0) / 60
    log(f"training done in {minutes:.1f} min | best epoch {best[2]} (mean dev accuracy {best[0]:.1%})")
    return {"train_minutes": minutes, "best_epoch": best[2]}

---
# Tests

## Cell 5 — One function per kind of test

As in `03`: every question scored on its own (one pass over all its options) and timed with the GPU synchronized. **Live routing** (E10, both independent sets, many-agent conversations), live and with the correct previous agent given; **first messages** (E11, both independent sets, many-agent), top-1 and top-2; **banking**; **questions** (E12, both independent sets, many-option). The **option-order check** routes the many-agent first messages again with the agents listed in reverse: the answer shouldn't change. Many-agent and many-option results are also shown by the number of options.

In [7]:
# Cell 5a — Timing, a table helper, and the test functions

def lat_summary(ms):
    ms = sorted(ms)
    return {"p50": ms[len(ms) // 2], "p95": ms[min(len(ms) - 1, int(0.95 * len(ms)))], "n": len(ms)}


def timed_probs(model, item):
    if AMP:
        torch.cuda.synchronize(DEVICE)
    t0 = time.perf_counter()
    p = model.probs(item)
    if AMP:
        torch.cuda.synchronize(DEVICE)
    return p, (time.perf_counter() - t0) * 1000


def table(rows, key, cols, groups=None):
    """Print right / total per group (key(row) -> group) and for ALL rows; cols = {column: row -> True if right}."""
    groups = list(groups or sorted({key(r) for r in rows})) + ["ALL"]
    w = max(len(str(g_)) for g_ in groups)
    print(f"  {'':{w}s} | {'n':>4} | " + " | ".join(f"{c:>12s}" for c in cols))
    for g_ in groups:
        rs = [r for r in rows if g_ == "ALL" or key(r) == g_]
        if rs:
            print(f"  {g_:{w}s} | {len(rs):>4} | " + " | ".join(f"{sum(map(f_, rs)):>5} / {len(rs):<4}" for f_ in cols.values()))


def size(n):
    return "2-5" if n <= 5 else "6-10" if n <= 10 else "11-15" if n <= 15 else "16-30" if n <= 30 else "31-50"


def user(text):
    return [{"role": "user", "content": text}]


def live_routing(model, convs, given=False):
    rows, lat = [], []
    for conv in convs:
        cur = None
        for u in conv["user_turns"]:
            p, ms = timed_probs(model, {"messages": conv["messages"][:u["message_index"] + 1], "current": cur, "question": ROUTE_Q,
                                        "options": conv["agents"]})
            pick = list(conv["agents"])[int(p.argmax())]
            rows.append({"company": conv.get("company"), "type": u["turn_type"], "size": size(len(conv["agents"])),
                         "ok": pick == u["expected_agent"]})
            lat.append(ms)
            cur = u["expected_agent"] if given else pick
    return rows, lat


def first_messages(model, msgs, reverse=False):
    rows, lat = [], []
    for m in msgs:
        names = list(m["agents"])[::-1] if reverse else list(m["agents"])
        p, ms = timed_probs(model, {"messages": user(m["message"]), "current": None, "question": ROUTE_Q,
                                    "options": {n: m["agents"][n] for n in names}})
        top = p.topk(2).indices.tolist()
        rows.append({"type": m["type"], "size": size(len(names)), "pick": names[top[0]], "ok": names[top[0]] == m["target_agent"],
                     "top2": m["target_agent"] in [names[i] for i in top]})
        lat.append(ms)
    return rows, lat


def banking_routing(model):
    rows, lat = [], []
    for ex in banking:
        p, ms = timed_probs(model, {"messages": ex["messages"], "current": ex["current_agent"], "question": ROUTE_Q, "options": BANKING_AGENTS})
        rows.append({"ok": list(BANKING_AGENTS)[int(p.argmax())] == ex["target_agent"]})
        lat.append(ms)
    return rows, lat


def questions(model, items):
    rows, lat = [], []
    for it in items:
        p, ms = timed_probs(model, {"messages": user(it["state"]), "current": None, "question": it["question"],
                                    "options": {o: "" for o in it["options"]}})
        top = int(p.argmax())
        rows.append({**it, "size": size(len(it["options"])), "pick": it["options"][top], "ok": it["options"][top] == it["answer"],
                     "conf": p.max().item(), "level_error": abs(top - it["answer_index"]) if it["type"] == "score" else None})
        lat.append(ms)
    return rows, lat

In [8]:
# Cell 5b — Run every test: R = {result: (right, total)}, LAT = {test: latency}

TYPE = {"noul": "yes/no", "choice": "choice", "score": "score", "many_choice": "many-option"}
right = {"right": lambda r: r["ok"]}
SIZES = ["2-5", "6-10", "11-15", "16-30", "31-50"]


def n_ok(rows, key="ok"):
    return sum(r[key] for r in rows), len(rows)


def evaluate(model):
    model.train(False)
    timed_probs(model, {"messages": user("warm-up"), "current": None, "question": ROUTE_Q, "options": BANKING_AGENTS})
    R, LAT = {}, {}
    for name, convs in [("E10", real_convs), ("IND", ind_r), ("IND2", ind2_r), ("MANY", many_r)]:
        log(f"{name}: routing every turn")
        rows, LAT[f"{name} routing"] = live_routing(model, convs)
        R[f"{name} routing live"], R[f"{name} routing given"] = n_ok(rows), n_ok(live_routing(model, convs, given=True)[0])
        if name == "E10":
            R["E10 test companies"] = n_ok([r for r in rows if r["company"] in TEST_COMPANIES])
        print(f"{name} routing live (given previous agent: {R[f'{name} routing given'][0]})")
        table(rows, lambda r: r["type"], right)
        if name in ("IND2", "MANY"):
            table(rows, lambda r: r["size"] + " agents", right, [s + " agents" for s in SIZES])
    for name, msgs in [("E11", e11), ("IND", ind_f), ("IND2", ind2_f), ("MANY", many_f)]:
        log(f"{name}: first messages")
        rows, LAT[f"{name} first"] = first_messages(model, msgs)
        R[f"{name} first top-1"], R[f"{name} first top-2"] = n_ok(rows), n_ok(rows, "top2")
        table(rows, lambda r: r["type"], {"top-1": lambda r: r["ok"], "top-2": lambda r: r["top2"]})
        if name == "MANY":
            rev, _ = first_messages(model, msgs, reverse=True)
            R["option order: same pick reversed"] = (sum(a["pick"] == b["pick"] for a, b in zip(rows, rev)), len(rows))
            print(f"  option order: same pick with the agents reversed {R['option order: same pick reversed'][0]} / {len(rows)} | "
                  f"top-1 forward {R['MANY first top-1'][0]}, reversed {n_ok(rev)[0]}")
    log("banking")
    rows, LAT["banking"] = banking_routing(model)
    R["banking 50"], R["banking 225"] = n_ok(rows[:50]), n_ok(rows)
    for name, items in [("E12", e12), ("IND", ind_q), ("IND2", ind2_q), ("MANY", many_q)]:
        log(f"{name}: questions")
        rows, LAT[f"{name} questions"] = questions(model, items)
        R[f"{name} questions"] = n_ok(rows)
        if name != "MANY":
            R.update({f"{name} {TYPE[t]}": n_ok([r for r in rows if r["type"] == t]) for t in TYPE if any(r["type"] == t for r in rows)})
            table(rows, lambda r: TYPE[r["type"]], right, [TYPE[t] for t in TYPE])
            sc = [r for r in rows if r["type"] == "score"]
            print(f"  score within 1 level: {sum(r['level_error'] <= 1 for r in sc)} / {len(sc)}")
        else:
            table(rows, lambda r: r["family"], right)
            table(rows, lambda r: r["size"] + " options", right, [s + " options" for s in SIZES])
        if name == "E12":
            print("  mistakes:")
            for r in [r for r in rows if not r["ok"]]:
                print(f"    {r['id']} {r['type']:6s} | {r['state'][:44]:44s} | {r['question'][:36]:36s} | "
                      f"expected {r['answer']:14s} | got {r['pick']} ({r['conf']:.2f})")
        if name == "IND":
            R["IND questions unflagged"] = n_ok([r for r in rows if not r.get("flag")])
    for name, L in LAT.items():
        L = lat_summary(L)
        print(f"latency {name:16s} p50 {L['p50']:4.0f} ms | p95 {L['p95']:4.0f} ms ({L['n']} questions)")
    return R, LAT

---
# The one-pass model

## Cell 6 — Train and test

**Expect** about 40 minutes of training on a T4 and run 28's results within the usual noise (±4 on most tests; more on the independent questions): E10 179 / 193 (76 at the test companies), E11 97, banking 49 · 212, E12 158, independent questions 221 and routing 134, independent 2 185 · 55 · 49; many-option 147 / 162, many-agent routing 100 / 120 and first messages 63 / 72, option order 68 / 72. Latency should stay flat at about 50–60 ms whatever the number of options.

In [9]:
# Cell 6 — Train (a collapse restarts with the next seed), then run every test

attempts = []
for attempt in range(3):
    seed = SEED + 1000 * attempt
    random.seed(seed); torch.manual_seed(seed)
    if AMP:
        torch.cuda.reset_peak_memory_stats(DEVICE)
    model = OnePassModel()
    log(f"attempt {attempt + 1} (seed {seed}) | head {sum(p.numel() for p in model.head.parameters()) / 1e6:.2f} M | {gpu()}")
    model.enc.print_trainable_parameters()
    try:
        trained = train_model(model, seed)
        attempts.append({"seed": seed, "status": "ok"})
        break
    except Collapsed as e:
        log(f"attempt {attempt + 1} COLLAPSED: {e} -> restarting with a new seed")
        attempts.append({"seed": seed, "status": f"collapsed: {e}"})
        model = None; gc.collect()                              # free the failed attempt
        torch.cuda.empty_cache() if AMP else None
else:
    raise RuntimeError(f"training collapsed in all attempts: {attempts}")

R, LAT = evaluate(model)
peak_gb = torch.cuda.max_memory_allocated(DEVICE) / 1e9 if AMP else 0
json.dump({"results": R, "latency": {k: lat_summary(v) for k, v in LAT.items()}, "attempts": attempts, **trained, "peak_gpu_gb": peak_gb},
          open("one_pass_results.json", "w"), indent=1)

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.58GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
decoder.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[07:34:48] attempt 1 (seed 42) | head 0.26 M | GPU 1.6 GB (peak 1.6)
trainable params: 7,196,672 || all params: 401,978,368 || trainable%: 1.7903
[07:34:48] training: 1184 batches per epoch (6319 routing + 11730 questions), 2368 steps
[07:36:03]   epoch 1 step 100/2368 | loss routing 1.6190 questions 1.0447 | 1.34 steps/s | ~28.3 min left | GPU 1.7 GB (peak 13.0)
[07:36:58]   epoch 1 step 200/2368 | loss routing 0.9433 questions 0.6431 | 1.54 steps/s | ~23.4 min left | GPU 1.7 GB (peak 13.0)
[07:38:08]   epoch 1 step 300/2368 | loss routing 0.3231 questions 0.4412 | 1.50 steps/s | ~23.0 min left | GPU 1.7 GB (peak 13.0)
[07:39:15]   epoch 1 step 400/2368 | loss routing 0.2671 questions 0.3915 | 1.50 steps/s | ~21.9 min left | GPU 1.7 GB (peak 13.0)
[07:40:29]   epoch 1 step 500/2368 | loss routing 0.2531 questions 0.4605 | 1.47 steps/s | ~21.2 min left | GPU 1.7 GB (peak 13.2)
[07:41:36]   epoch 1 step 600/2368 | loss routing 0.1305 questions 0.2521 | 1.47 steps/s | ~20.0 min left | GP

## Cell 7 — Save and verify

With `SAVE_MODEL = 1`, the model is written to the notebook's own working folder (on Kaggle, `/kaggle/working`, the Output panel); nothing is published. `adapter/` (the LoRA adapter alone, saved before merging), `encoder/` (merged ModernBERT in fp16 + tokenizer), `head.pt`, `signpost_config.json` (`"format": "one-pass"`, the input settings, this run's results) and the loader `signpost_one_pass.py`, zipped to `signpost-one-pass.zip`. The folder is then **reloaded with the loader** and E11, E12 and the many-option questions are scored again, in fp16 (GPU) and fp32 (what a CPU computes): they must match the trained model.

In [10]:
# Cell 7 — Save signpost-one-pass/ (+ adapter) and its zip, then reload with signpost_one_pass.py and score again

OUT = Path("signpost-one-pass")
if SAVE_MODEL:
    shutil.rmtree(OUT, ignore_errors=True)
    model.enc.save_pretrained(OUT / "adapter")                        # the LoRA adapter alone, before merging
    merged = model.enc.merge_and_unload()                             # W <- W + (alpha / r) B A in all 112 layers
    merged.half().save_pretrained(OUT / "encoder", safe_serialization=True)
    model.tok.save_pretrained(OUT / "encoder")
    torch.save({"state_dict": {k: v.float().cpu() for k, v in model.head.state_dict().items()}, "hidden": model.H}, OUT / "head.pt")
    cfg = {"name": "Signpost one-pass", "format": "one-pass", "base_model": ENCODER, "license_of_base": "Apache-2.0", "sep": SEP,
           "max_len": MAX_LEN, "max_options": MAX_OPTIONS, "route_question": ROUTE_Q, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
           "lora": f"peft {peft.__version__}", "score_loss": "plain", "epochs": EPOCHS, "best_epoch": trained["best_epoch"],
           "seed_attempts": attempts, "data_branch": DATA_BRANCH, "results": {k: v[0] for k, v in R.items()}}
    json.dump(cfg, open(OUT / "signpost_config.json", "w"), indent=1)
    shutil.copy(DATA_DIR.parent.parent / "signpost" / "signpost_one_pass.py", OUT / "signpost_one_pass.py")
    shutil.make_archive("signpost-one-pass", "zip", root_dir=".", base_dir=str(OUT))
    size_mb = {d: sum(f.stat().st_size for f in (OUT / d).rglob("*") if f.is_file()) / 1e6 for d in ("encoder", "adapter")}
    log(f"saved {OUT}/ (encoder {size_mb['encoder']:.0f} MB, adapter {size_mb['adapter']:.0f} MB) and signpost-one-pass.zip "
        f"({Path('signpost-one-pass.zip').stat().st_size / 1e9:.2f} GB)")

    sys.path.insert(0, str(OUT.resolve()))
    from signpost_one_pass import SignpostOnePass
    for dtype in (torch.float16, torch.float32):                     # fp16 = GPU use; fp32 = what a CPU run computes
        sp = SignpostOnePass(OUT, device=DEVICE, dtype=dtype)
        n11 = sum(sp.route(user(m["message"]), m["agents"])[0] == m["target_agent"] for m in e11)
        n12 = sum(sp.choice(it["state"], it["question"], it["options"])[0] == it["answer"] for it in e12)
        nmany = sum(sp.choice(it["state"], it["question"], it["options"])[0] == it["answer"] for it in many_q)
        log(f"reload check ({dtype}): E11 {n11}/{len(e11)} (trained model {R['E11 first top-1'][0]}) | E12 {n12}/{len(e12)} "
            f"(trained model {R['E12 questions'][0]}) | many-option {nmany}/{len(many_q)} (trained model {R['MANY questions'][0]})")
        del sp

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[08:06:31] saved signpost-one-pass/ (encoder 793 MB, adapter 29 MB) and signpost-one-pass.zip (0.76 GB)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[08:06:44] reload check (torch.float16): E11 98/100 (trained model 98) | E12 152/170 (trained model 152) | many-option 152/162 (trained model 152)


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

[08:07:06] reload check (torch.float32): E11 98/100 (trained model 98) | E12 152/170 (trained model 152) | many-option 152/162 (trained model 152)


---
# Results

## Cell 8 — Side by side

The other columns are measured results on the same items, not run here: **run 28** (this recipe's previous run, `exp/listwise-index`); the **published Signpost** (run 25 on the old tests; the Hub model on independent set 2 and the many-option tests, measured in run 28's notebook); the **decider** (`04`; many-option tests in run 28's notebook). The decider has no current-agent input, so "given" doesn't apply to it. Latency is p50 in ms on a T4: Signpost from `03`'s final run and the Hub model, the decider from `04` and run 28's notebook.

In [11]:
# Cell 8 — This run next to run 28, the published Signpost and the decider, then latency

REF = {
    "run 28": {"E10 routing live": 179, "E10 test companies": 76, "IND routing live": 134, "IND2 routing live": 55, "MANY routing live": 100,
               "E11 first top-1": 97, "IND first top-1": 180, "IND2 first top-1": 49, "MANY first top-1": 63, "banking 50": 49,
               "banking 225": 212, "E12 questions": 158, "E12 yes/no": 57, "E12 choice": 56, "E12 score": 45, "IND questions": 221,
               "IND yes/no": 70, "IND choice": 105, "IND score": 46, "IND2 questions": 185, "IND2 yes/no": 44, "IND2 choice": 47,
               "IND2 score": 46, "IND2 many-option": 48, "MANY questions": 147, "option order: same pick reversed": 68},
    "Signpost (published)": {"E10 routing live": 181, "E10 test companies": 78, "IND routing live": 133, "IND2 routing live": 62,
                             "IND2 routing given": 65, "MANY routing live": 103, "MANY routing given": 109, "E11 first top-1": 97,
                             "IND first top-1": 180, "IND2 first top-1": 47, "MANY first top-1": 62, "banking 50": 48, "banking 225": 210,
                             "E12 questions": 161, "E12 yes/no": 58, "E12 choice": 58, "E12 score": 45, "IND questions": 221,
                             "IND yes/no": 75, "IND choice": 103, "IND score": 43, "IND2 questions": 188, "IND2 yes/no": 47,
                             "IND2 choice": 47, "IND2 score": 45, "IND2 many-option": 49, "MANY questions": 150},
    "decider 2B": {"E10 routing live": 151, "E10 test companies": 62, "IND routing live": 113, "IND2 routing live": 55, "MANY routing live": 83,
                   "E11 first top-1": 96, "E11 first top-2": 100, "IND first top-1": 178, "IND first top-2": 193, "IND2 first top-1": 50,
                   "MANY first top-1": 70, "banking 50": 48, "banking 225": 214, "E12 questions": 160, "E12 yes/no": 56, "E12 choice": 56,
                   "E12 score": 48, "IND questions": 253, "IND yes/no": 77, "IND choice": 106, "IND score": 70, "IND questions unflagged": 253,
                   "IND2 questions": 185, "IND2 yes/no": 49, "IND2 choice": 46, "IND2 score": 43, "IND2 many-option": 47, "MANY questions": 160},
}
P50 = {"Signpost": {"E10 routing": 58, "IND routing": 70, "IND2 routing": 187, "MANY routing": 99, "E11 first": 57, "IND first": 57,
                    "IND2 first": 73, "MANY first": 51, "banking": 73, "E12 questions": 55, "IND questions": 55, "IND2 questions": 57,
                    "MANY questions": 63},
       "decider": {"E10 routing": 316, "IND routing": 376, "IND2 routing": 654, "MANY routing": 574, "E11 first": 297, "IND first": 287,
                   "IND2 first": 463, "MANY first": 465, "banking": 468, "E12 questions": 197, "IND questions": 199, "IND2 questions": 200,
                   "MANY questions": 382}}

print(f"{'result':34s} | {'this run':>11} | " + " | ".join(f"{c:>20s}" for c in REF))
for k, (ok_, n) in R.items():
    print(f"{k:34s} | {f'{ok_} / {n}':>11} | " + " | ".join(f"{REF[c].get(k, '—'):>20}" for c in REF))
print(f"\n{'latency p50 (ms)':18s} | {'this run':>8} | {'Signpost':>8} | {'decider':>8}")
for k, v in LAT.items():
    print(f"{k:18s} | {lat_summary(v)['p50']:>8.0f} | {P50['Signpost'].get(k, '—'):>8} | {P50['decider'].get(k, '—'):>8}")
print(f"\ntraining {trained['train_minutes']:.1f} min, best epoch {trained['best_epoch']} | attempts {[a['status'] for a in attempts]} | "
      f"peak GPU {peak_gb:.1f} GB")

result                             |    this run |               run 28 | Signpost (published) |           decider 2B
E10 routing live                   |   175 / 193 |                  179 |                  181 |                  151
E10 routing given                  |   179 / 193 |                    — |                    — |                    —
E10 test companies                 |     76 / 83 |                   76 |                   78 |                   62
IND routing live                   |   134 / 172 |                  134 |                  133 |                  113
IND routing given                  |   138 / 172 |                    — |                    — |                    —
IND2 routing live                  |     56 / 70 |                   55 |                   62 |                   55
IND2 routing given                 |     59 / 70 |                    — |                   65 |                    —
MANY routing live                  |   102 / 120 |      

---
## Cell 9 — What we tried and dropped

Every number is correct / total on the same test items; the runs are in `experiments/modernbert_router_ledger.md` (runs 26–29, branch `exp/listwise-index`). One seed per run; run-to-run noise is about ±4 on most tests.

| Run | What we tried | Result | Decision |
|---|---|---|---|
| 26 | One pass per question, with **30% of Signpost's routing and question data** + the 2,500 many-option questions | Level with Signpost on most tests at a flat ~51–55 ms for 10–50 options; fresh tests: many-option 151 / 162 (Signpost 150, decider 160), many-agent routing 104 / 120 (103, decider 83). But E10 all −9 and E12 −7 | More data |
| 27 | **All of Signpost's data** + many-option questions + 2,500 routing items trained with 10–15 agents; ordinal score loss | Routing fixed (E10 all 177, banking 211, independent routing 136), but independent score fell to 35 / 92 and E12 to 153 | The ordinal loss hurt score here |
| 28 | The same with **plain cross-entropy** for score questions | Independent score 46 (questions 221 = Signpost), E12 158; level with Signpost on every old test (within ±3) at ~55–60 ms | **Kept: this notebook's recipe** |
| 29 | **No score questions** in training or tests | Nothing improved; E12 without score 113 → 107 and independent routing 134 → 128 | Dropped: score questions help the other jobs |

**What one pass buys:** latency stays flat however many options there are (about 55 ms for 50 options, where Signpost's grows with each one: about 190 ms with 8–11 agents), and the options can be compared inside the encoder. **What it doesn't:** accuracy beyond Signpost's; both share the score weakness on outside-written scales.

---
## Next

The saved model can be loaded with `signpost/signpost_one_pass.py` (same calls as `signpost.py`: `route`, `yes_no`, `choice`, `score`). Whether it is published next to Signpost is a separate decision; nothing here uploads it.